# Render Blender di Colab (GPU gratis)

Urutan: setel runtime **GPU** dulu (Runtime > Change runtime type > T4 GPU), lalu jalankan sel dari atas ke bawah.

Yang dibutuhkan: file `blender-scene.zip` (isi folder `scene/` dari farm kit: scene.py + tekstur PNG + composite.py).

In [ ]:
!nvidia-smi

In [ ]:
!apt-get update -qq && apt-get install -y -qq blender xvfb libegl1 libegl-mesa0 libgl1-mesa-dri libgles2 fonts-dejavu-core > /dev/null 2>&1
!blender --version | head -1

In [ ]:
from google.colab import files
up = files.upload()  # pilih blender-scene.zip
!rm -rf /content/scene && mkdir -p /content/scene && unzip -qo blender-scene.zip -d /content/scene && ls /content/scene

In [ ]:
# Generate tekstur PNG (sengaja tidak disimpan di repo/zip — dibuat dari kode)
!cd /content/scene && python3 make_sign.py

In [ ]:
# Render semua frame (ganti rentang untuk chunk manual, mis. 1 120 / 121 240 di dua sesi)
import os
os.environ['RENDER_PCT'] = '50'  # 540x960; naikkan ke 100 untuk native 1080x1920
!cd /content/scene && xvfb-run -a blender -b -P scene.py -- anim 1 240 2>&1 | tail -3
!ls /content/scene/frames | wc -l

In [ ]:
# Pilihan A: unduh frame saja (komposit + encode dilakukan di VM Muse)
!cd /content/scene && zip -qr frames.zip frames
files.download('/content/scene/frames.zip')

In [ ]:
# Pilihan B: sekalian jadi video di sini
!pip install --quiet pillow
!apt-get install -y -qq ffmpeg > /dev/null 2>&1
!cd /content/scene && python3 composite.py && ls -la blender-proof-01.mp4
files.download('/content/scene/blender-proof-01.mp4')